In [1]:
#Author: Hiba Ali 
#E-mail: hhibaa802@gmail.com 
#Purpose: QSS Thesis

In [3]:
#import libraries 
import pandas as pd
import os

In [4]:
#read data
df = pd.read_csv("../data/politicians_list.csv") 
df.head()

,FirstName,LastName,X Handle,St/Dis,Party,Role,Past or current?
0,Alma,Adams,@RepAdams,NC12,D,Representative,Current
1,Robert,Aderholt,@Robert_Aderholt,AL04,R,Representative,Current
2,Pete,Aguilar,@RepPeteAguilar,CA33,D,Representative,Current
3,Mark,Alford,@RepMarkAlford,MO04,R,Representative,Current
4,Rick,Allen,@RepRickAllen,GA12,R,Representative,Current


In [5]:
#check shape
df.shape

#735 users total

(735, 7)

In [6]:
#check for duplicates
duplicates = df.duplicated(subset=['FirstName', 'LastName'], keep=False)

#show rows that are duplicate
df[duplicates]

,FirstName,LastName,X Handle,St/Dis,Party,Role,Past or current?
74,Lou,Correa,@RepLouCorrea,CA46,D,Representative,Current
159,Morgan,Griffith,@RepMGriffith,VA09,R,Representative,Current
438,Jim,Banks,@SenatorBanks,IN,R,Senate,Current
499,Markwayne,Mullin,@SenMullin,OK,R,Senate,Current
521,Elissa,Slotkin,@SenatorSlotkin,MI,D,Senate,Current
644,Elissa,Slotkin,@SenatorSlotkin,NaN,D,Representative,Past
654,Morgan,Griffith,@RepMGriffith,NaN,R,Representative,Past
656,Lou,Correa,@RepLouCorrea,NaN,D,Representative,Past
669,Jim,Banks,@Jim_Banks,NaN,R,Representative,Past
695,Markwayne,Mullin,@MarkwayneMullin,NaN,R,Representative,Past


In [7]:
df = df.dropna(subset=['X Handle'])
df.shape #now have 723 politicians 

(723, 7)

In [13]:
#drop duplicates

# Drop rows with specific indices
rows_to_drop = [644, 654, 656, 669, 695]
df = df.drop(index=rows_to_drop)

# Optional: reset the index
df = df.reset_index(drop=True)

print(df.shape)
#now have 718 politicians)

(718, 7)


In [15]:
df['Role'].value_counts()

Role
Representative                                             570
Senate                                                     100
Administrator of the Small Business Administration           2
White House Chief of Staff                                   2
President of the United States                               2
Vice President of the United States                          2
U.S. Trade Representative                                    1
U.S. Secretary of Education                                  1
Director of the Office of Science and Technology Policy      1
Director of the U.S. Office of Management and Budget         1
U.S. Ambassador to the United Nations                        1
U.S. Secretary of Agriculture                                1
U.S. Secretary of Commerce                                   1
U.S. Secretary of Defense                                    1
U.S. Secretary of Energy                                     1
U.S. Secretary of Veterans Affairs                

In [17]:
# Separate the groups
sen_rep = df[df['Role'].isin(['Senator', 'Representative'])]
other = df[~df['Role'].isin(['Senator', 'Representative'])]

# Randomly sample 202 Senators/Representatives out of the 670 total, about 30%
sen_rep_sample = sen_rep.sample(n=202, random_state=42)

# Combine with all other roles
final_df = pd.concat([sen_rep_sample, other], ignore_index=True)

print(final_df.shape)  # Should be 350

(350, 7)


In [19]:
# Remove '@' from 'X Handle' column
final_df['X Handle'] = final_df['X Handle'].str.replace('@', '', regex=False)

In [35]:
final_df.to_csv("final_df.csv", index=False)

In [ ]:
PULLING HEALTH-RELATED TWEETS AND SAVING THEM

In [27]:
import requests
import time
import json

API_BASE_URL = "https://api.data365.co"
API_VERSION = "/v1.1"
API_AUTHORIZATION_TOKEN = "ZXlKMGVYQWlPaUpLVjFRaUxDSmhiR2NpT2lKSVV6STFOaUo5LmV5SnpkV0lpT2lKSWFXSmhRV3hwSWl3aWFXRjBJam94TnpZNU5UQTNOamcyTGpBd05UQXhPSDAuNmpTLTZtbVBGV0F1ZS1McHhXaUIwbUtDUWFDemU4VkRNZVhOSkZ2ckJRTQ=="

In [37]:
# Map shortcut numbers to keyword blocks
KEYWORD_BLOCKS = {
    "1A": "health OR medicine OR medical OR medication",
    "1B": "treatment OR symptoms OR diagnosis",
    "2A": "healthcare OR insurance OR medicare OR medicaid",
    "2B": "premiums OR coverage OR patients OR hospitals",
    "3": "flu OR pneumonia OR bronchitis OR strep throat OR sore throat OR cough OR fever OR infection OR contagious",
    "4": "high blood pressure OR hypertension OR heart OR heart disease OR heart rate OR stroke OR cholesterol OR diabetes OR cancer",
    "5": "lupus OR autoimmune OR chronic illness OR inflammation OR immune system",
    "6": "hydration OR diet OR nutrition OR exercise OR workout OR sleep OR wellness"
}


In [467]:
# Choose your shortcut number for this run
shortcut = "6"  
keywords = KEYWORD_BLOCKS[shortcut]

# Start the search task
update_url = f"{API_BASE_URL}{API_VERSION}/twitter/search/post/update"

params = {
    "from_profile": "michaelgwaltz",
    "from_date": "2022-01-01",
    "to_date": "2025-12-31",
    "lang": "en",
    "max_posts": 20,
    "search_type": "latest",
    "max_replies": 3,
    "load_replies": "true",
    "keywords": keywords,
    "access_token": API_AUTHORIZATION_TOKEN
}

# POST to start task
post_response = requests.post(update_url, params=params)
print("Task started:", post_response.status_code)

# GET the actual posts
posts_url = f"{API_BASE_URL}{API_VERSION}/twitter/search/post/posts"
posts_response = requests.get(posts_url, params=params)

if posts_response.status_code == 200:
    data = posts_response.json()
    print("Number of items retrieved:", len(data.get("data", {}).get("items", [])))
else:
    print("Failed to fetch posts:", posts_response.status_code)

Task started: 202
Number of items retrieved: 3


In [419]:
print(json.dumps(data, indent=4))

{
    "data": {
        "items": [
            {
                "attached_links_expanded_url": null,
                "attached_links_url": null,
                "attached_medias_preview_url": [
                    "https://pbs.twimg.com/amplify_video_thumb/1976699604214112256/img/JCGmc8M5DnuaK6q6.jpg?name=small"
                ],
                "attached_medias_url": [
                    "https://video.twimg.com/amplify_video/1976699604214112256/vid/avc1/1920x1080/_mCCXqu5DmR651Jq.mp4?tag=21"
                ],
                "attached_videos": [
                    {
                        "duration": 304,
                        "id": "1976699604214112256",
                        "url": "https://video.twimg.com/amplify_video/1976699604214112256/vid/avc1/1920x1080/_mCCXqu5DmR651Jq.mp4?tag=21",
                        "view_count": null
                    }
                ],
                "author_id": "1874221699253616640",
                "author_username": "RepSheriBiggs",

In [469]:
politician_name = "michaelgwaltz"   
block_name = shortcut          

filename = f"{politician_name} Block {block_name}.json"

with open(filename, "w", encoding="utf-8") as f:
    json.dump(data, f, indent=4)

print(f"Saved results to: {filename}")

Saved results to: michaelgwaltz Block 6.json


In [ ]:
PULLING HEALTH-RELATED TWEET REPLIES AND SAVING THEM

In [423]:
username = "RepWexton"
input_filename = "RepWexton Block 1A.json"
output_filename = "RepWexton Block 1A Replies.json"

# ------------------------------------
# 1. Open the saved block file
# ------------------------------------
with open(input_filename, "r", encoding="utf-8") as f:
    block_data = json.load(f)

posts = block_data.get("data", {}).get("items", [])

all_replies_output = []

# ------------------------------------
# 2. Extract every conversation_id
# ------------------------------------
for post in posts:
    conversation_id = post.get("conversation_id")

    if not conversation_id:
        continue

    print(f"Fetching replies for post {conversation_id}")

    # ------------------------------------
    # 3. Call replies endpoint
    # ------------------------------------
    replies_url = f"{API_BASE_URL}{API_VERSION}/twitter/profile/{username}/feed/posts/{conversation_id}/replies"

    params = {
        "order_by": "date_desc",
        "max_page_size": 50,
        "access_token": API_AUTHORIZATION_TOKEN
    }

    response = requests.get(replies_url, params=params)

    if response.status_code == 200:
        replies_json = response.json()
        replies_items = replies_json.get("data", {}).get("items", [])

        all_replies_output.append({
            "conversation_id": conversation_id,
            "replies": replies_items
        })

    else:
        print(f"Failed to fetch replies for {conversation_id}: {response.status_code}")

    time.sleep(1)  # small pause to avoid rate limits

# ------------------------------------
# 4. Save everything into new JSON file
# ------------------------------------
final_output = {
    "politician": username,
    "block": "1A",
    "posts": all_replies_output
}

with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(final_output, f, indent=4)

print("Replies file successfully created.")

Fetching replies for post 1681346607915171840
Fetching replies for post 1691853850901184714
Fetching replies for post 1703741063586857406
Fetching replies for post 1706358464434332052
Fetching replies for post 1707063663046127705
Fetching replies for post 1709240893574721830
Fetching replies for post 1711848831409656123
Fetching replies for post 1714437316641497384
Fetching replies for post 1716523703024959606
Fetching replies for post 1716860681834635567
Fetching replies for post 1720193384449458260
Fetching replies for post 1735391554926252269
Fetching replies for post 1742603450469449968
Fetching replies for post 1744430875243147448
Fetching replies for post 1745161898536874232
Fetching replies for post 1745574661444767914
Fetching replies for post 1749502018228465933
Fetching replies for post 1762156055448936843
Fetching replies for post 1765443977522643353
Fetching replies for post 1765443977522643353
Fetching replies for post 1765930136275132816
Fetching replies for post 17661943